# Setup and the course LLM client

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /home/jsepu/LanternCS2 | LLM mode: replay


Run this notebook first. It checks your environment, seeds the replay cache from the recorded responses in `data/llm_fixtures/`, and shows the client from the orientation slides (parts 1 to 4) at work. Replay mode needs no API key.

In [2]:
import importlib
for pkg in ["yaml", "numpy", "pandas", "sklearn", "openpyxl", "bs4", "pydantic", "duckdb", "chromadb", "rank_bm25", "langchain_text_splitters", "fastapi", "matplotlib"]:
    try:
        importlib.import_module(pkg); print(f"ok       {pkg}")
    except ImportError:
        print(f"MISSING  {pkg}  ->  pip install -r requirements.txt")

ok       yaml
ok       numpy
ok       pandas


ok       sklearn
ok       openpyxl
ok       bs4
ok       pydantic
ok       duckdb


ok       chromadb
ok       rank_bm25


ok       langchain_text_splitters
ok       fastapi
ok       matplotlib


In [3]:
import subprocess
subprocess.run([sys.executable, "tools/seed_replay_cache.py"], check=True)

seeded 12 model responses into data/llm_cache and 37 embeddings into data/embed_cache


CompletedProcess(args=['/home/jsepu/LanternCS2/.venv/bin/python', 'tools/seed_replay_cache.py'], returncode=0)

`src/llm/client.py` is the slides' code (parts 1, 3, 2, 4 in dependency order) plus a small adapter for live calls. Every model and embedding call in Case Study 2 goes through it.

In [4]:
import inspect
print(inspect.getsource(llm.structured))

def structured(response_model, messages, model,
               stage="build", **kw):
    schema = response_model.model_json_schema()
    key = cache_key(PROVIDER, model, messages,
                    schema=schema, **kw)
    f = CACHE / f"{key}.json"
    if f.exists():                          # hit
        return response_model.model_validate_json(
            f.read_text())
    if MODE == "replay":
        raise RuntimeError(f"cache miss {key}")
    # budget_check and ledger are defined in part 3
    budget_check(model, len(json.dumps(messages)) // 4)
    out, raw = INSTR.create_with_completion(
        response_model=response_model, model=model,
        messages=messages, **kw)
    if MODE == "record":          # cache before ledger,
        f.write_text(out.model_dump_json())  # so no re-pay
    u = raw.usage                 # names vary by provider
    ledger(model, u.prompt_tokens,
           u.completion_tokens, stage)
    return out



In [5]:
# A replay hit: the recorded claim extraction for one CIM block
blocks = {r["block_id"]: r for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # block ids are unique only within a document
out = llm.structured(schemas.MgmtClaims, prompts.claims_messages(blocks["p0002_b001"]["text"]), llm.P["llm"]["models"]["large"])
for c in out.claims: print(c.metric, c.value, c.unit, c.period, c.qualifier)

revenue_growth 0.32 pct FY2025 reported
revenue 41.2 USD FY2025 reported


In [6]:
# A replay miss is an error that names the missing key: replay never calls a provider
try:
    llm.structured(schemas.MgmtClaims, prompts.claims_messages("A sentence nobody recorded."), llm.P["llm"]["models"]["large"])
except RuntimeError as e:
    print("as expected:", e)

as expected: cache miss 84864a8fa130c8583aa5049ca783e8d6554a68b366909fc8366eda7361205371


In [7]:
# The cost ledger and the budget cap (part 3): what a live call of 1,000 tokens in and 300 out would cost
print("USD:", llm.usd(llm.P["llm"]["models"]["large"], 1000, 300), "| cap:", llm.CAP, "| spent so far:", llm.spent())

USD: 0.00088 | cap: 5.0 | spent so far: 0.00046100000000000004


In [8]:
# Is the cache key canonical? (handout p. 14): dict order must not matter, one extra space must
k1 = llm.cache_key("openai", "gpt-4.1-mini", [{"role": "user", "content": "hi"}])
k2 = llm.cache_key("openai", "gpt-4.1-mini", [{"content": "hi", "role": "user"}])
k3 = llm.cache_key("openai", "gpt-4.1-mini", [{"role": "user", "content": "hi "}])
print(k1 == k2, k1 == k3, k1[:12])

True False 64a92e5118fe


**Exercise 0.** Switch to record mode with your own key (`export LANTERN_LLM_MODE=record` and `OPENAI_API_KEY=...`), ask one new extraction, and confirm that a cache file and one ledger row appear. Then switch back to replay and run it again offline.

In [9]:
# Exercise 0: In [6] with my own sentence. Record mode: makes the call and caches it. Replay mode: served from the cache
try:
    llm.structured(schemas.MgmtClaims, prompts.claims_messages("ARR reached $12 million in FY2025."), llm.P["llm"]["models"]["large"])
except RuntimeError as e:
    print("as expected:", e)